In [47]:
import pandas as pd
import numpy as np

In [48]:
df = pd.read_csv("C:/Procurement-Supplier-Performance-Analytics/data/cleaned/procurement_cleaned.csv")

In [49]:
df.head()

,transaction_type,actual_shipping_days,scheduled_shipping_days,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_region,order_state,order_status,product_id,product_category_id,product_name,product_price,product_status,shipping_date,shipping_mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,Southeast Asia,Java Occidental,COMPLETE,1360,73,Smart watch,327.75,0,2018-03-02 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,South Asia,Rajastán,PENDING,1360,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,South Asia,Rajastán,CLOSED,1360,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,Oceania,Queensland,COMPLETE,1360,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,Oceania,Queensland,PENDING_PAYMENT,1360,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class


In [50]:
rename_columns = {
    "Days for shipping (real)": "actual_shipping_days",
    "Days for shipment (scheduled)": "scheduled_shipping_days",
    "Product Price": "product_price",
    "Order Item Quantity": "quantity",
    "Category Name": "category",
    "Order Id": "order_id",
    "order date (DateOrders)": "order_date",
    "shipping date (DateOrders)": "shipping_date",
    "Delivery Status": "delivery_status",
    "Shipping Mode": "shipping_mode",
    "Order Country": "order_country",
    "Order State": "order_state",
    "Order City": "order_city",
    "Order Region": "order_region",
    "Customer Segment": "customer_segment",
    "Market": "market",
    "Product Name": "product_name",
    "Order Item Product Price": "unit_price",
    "Order Item Total": "total_price"
}


In [51]:
df = df.rename(columns=rename_columns)

In [52]:
df.columns.tolist()

['transaction_type',
 'actual_shipping_days',
 'scheduled_shipping_days',
 'benefit_per_order',
 'sales_per_customer',
 'delivery_status',
 'late_delivery_risk',
 'category_id',
 'category_name',
 'customer_city',
 'customer_country',
 'customer_id',
 'customer_segment',
 'customer_state',
 'department_id',
 'department_name',
 'market',
 'order_city',
 'order_country',
 'order_customer_id',
 'order_date',
 'order_id',
 'order_item_cardprod_id',
 'order_item_discount',
 'order_item_discount_rate',
 'order_item_id',
 'unit_price',
 'order_item_profit_ratio',
 'quantity',
 'sales',
 'order_item_total',
 'order_profit_per_order',
 'order_region',
 'order_state',
 'order_status',
 'product_id',
 'product_category_id',
 'product_name',
 'product_price',
 'product_status',
 'shipping_date',
 'shipping_mode']

In [53]:
df["lead_time"] = df["actual_shipping_days"]

In [54]:
df[["actual_shipping_days", "lead_time"]].head()

,actual_shipping_days,lead_time
0,3,3
1,5,5
2,4,4
3,3,3
4,2,2


In [55]:
df["delivery_delay"] = (
    df["actual_shipping_days"] -
    df["scheduled_shipping_days"]
)

In [56]:
conditions = [
    df["delivery_delay"] < 0,
    df["delivery_delay"] == 0,
    df["delivery_delay"] > 0
]


In [57]:
choices = [
    "Early",
    "On Time",
    "Late"
]


In [58]:
df["delivery_performance"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

In [59]:
np.random.seed(42)

cost_factor = np.random.uniform(
    0.60,
    0.85,
    len(df)
)

In [60]:
df["procurement_cost"] = (
    df["product_price"] *
    cost_factor *
    df["quantity"]
).round(2)

In [61]:
df["procurement_spend"] = df["procurement_cost"]

In [65]:
import numpy as np

np.random.seed(42)  # Ensures reproducible results

suppliers = [
    "Alpha Industrial Supplies",
    "Global Procurement Ltd",
    "Prime Source Traders",
    "Vertex Supply Co.",
    "BlueRock Manufacturing",
    "Evergreen Wholesale",
    "Nexus Components",
    "Titan Distributors",
    "Apex Procurement",
    "Sigma Global Supply",
    "Pioneer Industries",
    "Unity Wholesale",
    "Horizon Supply Chain",
    "Nova Traders",
    "Summit Industrial"
]

In [66]:
category_supplier = {}

categories = df["category_name"].unique()

for i, cat in enumerate(categories):
    category_supplier[cat] = suppliers[i % len(suppliers)]

df["supplier_name"] = df["category_name"].map(category_supplier)

In [67]:
supplier_ids = {
    supplier: f"SUP{str(i + 1).zfill(3)}"
    for i, supplier in enumerate(df["supplier_name"].unique())
}

df["supplier_id"] = df["supplier_name"].map(supplier_ids)

In [68]:
ratings = {
    supplier: round(np.random.uniform(3.5, 5.0), 1)
    for supplier in supplier_ids
}

df["supplier_rating"] = df["supplier_name"].map(ratings)

In [69]:
contract = [
    "Annual",
    "Quarterly",
    "Spot Purchase"
]

contracts = {
    supplier: np.random.choice(contract)
    for supplier in supplier_ids
}

df["contract_type"] = df["supplier_name"].map(contracts)

In [70]:
warehouses = [
    "Mumbai",
    "Delhi",
    "Bengaluru",
    "Hyderabad",
    "Chennai"
]

df["warehouse"] = np.random.choice(
    warehouses,
    len(df)
)

In [71]:
df["inventory_level"] = np.random.randint(
    50,
    500,
    len(df)
)

In [72]:
df["reorder_level"] = np.random.randint(
    40,
    120,
    len(df)
)

In [73]:
df["safety_stock"] = np.random.randint(
    20,
    80,
    len(df)
)

In [74]:
df["inventory_status"] = np.where(
    df["inventory_level"] <= df["reorder_level"],
    "Reorder Required",
    "Sufficient"
)

In [75]:
payment_status = [
    "Paid",
    "Pending",
    "Overdue"
]

df["payment_status"] = np.random.choice(
    payment_status,
    len(df),
    p=[0.75,0.20,0.05]
)

In [76]:
df["procurement_cost"] = (
    df["product_price"] *
    df["quantity"] *
    np.random.uniform(0.60, 0.85, len(df))
).round(2)

In [77]:
df["vendor_performance_score"] = (
    (df["supplier_rating"] * 20)
    -
    (df["delivery_delay"] * 5)
)

df["vendor_performance_score"] = (
    df["vendor_performance_score"]
    .clip(0,100)
    .round(1)
)

In [78]:
new_columns = [
    "supplier_id",
    "supplier_name",
    "supplier_rating",
    "contract_type",
    "warehouse",
    "inventory_level",
    "reorder_level",
    "safety_stock",
    "inventory_status",
    "payment_status",
    "procurement_cost",
    "vendor_performance_score"
]

print(df[new_columns].head())

  supplier_id              supplier_name  supplier_rating contract_type  \
0      SUP001  Alpha Industrial Supplies              4.1        Annual   
1      SUP001  Alpha Industrial Supplies              4.1        Annual   
2      SUP001  Alpha Industrial Supplies              4.1        Annual   
3      SUP001  Alpha Industrial Supplies              4.1        Annual   
4      SUP001  Alpha Industrial Supplies              4.1        Annual   

   warehouse  inventory_level  reorder_level  safety_stock inventory_status  \
0  Bengaluru              442             89            78       Sufficient   
1  Hyderabad              198             48            75       Sufficient   
2  Hyderabad              145             58            27       Sufficient   
3     Mumbai              333             72            36       Sufficient   
4  Bengaluru              448            117            21       Sufficient   

  payment_status  procurement_cost  vendor_performance_score  
0        Pe

In [79]:
df.to_csv(
    "C:/Procurement-Supplier-Performance-Analytics/data/cleaned/procurement_feature_engineered.csv", index=False)

In [80]:
suppliers = (
    df[
        [
            "supplier_id",
            "supplier_name",
            "supplier_rating",
            "contract_type"
        ]
    ]
    .drop_duplicates()
    .sort_values("supplier_id")
    .reset_index(drop=True)
)

In [ ]:
suppliers.to_csv("C:/Procurement-Supplier-Performance-Analytics/data/sql/suppliers.csv", index=False)

In [82]:
print(suppliers.shape)
suppliers.head()

(15, 4)


,supplier_id,supplier_name,supplier_rating,contract_type
0,SUP001,Alpha Industrial Supplies,4.1,Annual
1,SUP002,Global Procurement Ltd,4.9,Annual
2,SUP003,Prime Source Traders,4.6,Quarterly
3,SUP004,Vertex Supply Co.,4.4,Quarterly
4,SUP005,BlueRock Manufacturing,3.7,Annual


In [83]:
product_categories = (
    df[
        [
            "category_id",
            "category_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("category_id")
    .reset_index(drop=True)
)

product_categories.to_csv(
    "C:/Procurement-Supplier-Performance-Analytics/data/sql/product_categories.csv",
    index=False
)

In [84]:
print(product_categories.shape)
product_categories.head()

(51, 2)


,category_id,category_name
0,2,Soccer
1,3,Baseball & Softball
2,4,Basketball
3,5,Lacrosse
4,6,Tennis & Racquet


In [101]:
products = (
    df[
        [
            "product_id",
            "product_name",
            "category_id",
            "category_name",
            "product_price",
            "supplier_id"
        ]
    ]
    .drop_duplicates(subset=["product_id"])
    .sort_values("product_id")
    .reset_index(drop=True)
)

# round to 2 decimal places to match DECIMAL(10,2)
products["product_price"] = products["product_price"].round(2)

products.to_csv(
    "C:/Procurement-Supplier-Performance-Analytics/data/sql/products.csv",
    index=False
)

In [96]:
print(products.shape)
products.head()

(118, 6)


,product_id,product_name,category_id,category_name,product_price,supplier_id
0,19,Nike Men's Fingertrap Max Training Shoe,2,Soccer,124.989998,SUP003
1,24,Elevation Training Mask 2.0,2,Soccer,79.989998,SUP003
2,35,adidas Brazuca 2014 Official Match Ball,3,Baseball & Softball,159.990005,SUP011
3,37,adidas Kids' F5 Messi FG Soccer Cleat,3,Baseball & Softball,34.990002,SUP011
4,44,adidas Men's F10 Messi TRX FG Soccer Cleat,3,Baseball & Softball,59.990002,SUP011


In [104]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 118 entries, 0 to 117
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     118 non-null    int64  
 1   product_name   118 non-null    str    
 2   category_id    118 non-null    int64  
 3   category_name  118 non-null    str    
 4   product_price  118 non-null    float64
 5   supplier_id    118 non-null    str    
dtypes: float64(1), int64(2), str(3)
memory usage: 5.7 KB


In [102]:
purchase_orders = df[
    [
        "order_item_id",
        "order_id",
        "order_date",
        "shipping_date",
        "product_id",
        "quantity",
        "unit_price",
        "procurement_cost",
        "lead_time",
        "delivery_delay",
        "delivery_status",
        "shipping_mode",
        "order_country",
        "order_region"
    ]
]
# round both cost columns to match DECIMAL(x,2) in MySQL
purchase_orders["unit_price"] = purchase_orders["unit_price"].round(2)
purchase_orders["procurement_cost"] = purchase_orders["procurement_cost"].round(2)

purchase_orders.to_csv(
    "C:/Procurement-Supplier-Performance-Analytics/data/sql/purchase_orders.csv",
    index=False
)

In [103]:
print(purchase_orders.shape)
purchase_orders.head()

(180519, 14)


,order_item_id,order_id,order_date,shipping_date,product_id,quantity,unit_price,procurement_cost,lead_time,delivery_delay,delivery_status,shipping_mode,order_country,order_region
0,180517,77202,2018-01-31 22:56:00,2018-03-02 22:56:00,1360,1,327.75,231.56,3,-1,Advance shipping,Standard Class,Indonesia,Southeast Asia
1,179254,75939,2018-01-13 12:27:00,2018-01-18 12:27:00,1360,1,327.75,271.57,5,1,Late delivery,Standard Class,India,South Asia
2,179253,75938,2018-01-13 12:06:00,2018-01-17 12:06:00,1360,1,327.75,237.26,4,0,Shipping on time,Standard Class,India,South Asia
3,179252,75937,2018-01-13 11:45:00,2018-01-16 11:45:00,1360,1,327.75,276.96,3,-1,Advance shipping,Standard Class,Australia,Oceania
4,179251,75936,2018-01-13 11:24:00,2018-01-15 11:24:00,1360,1,327.75,216.44,2,-2,Advance shipping,Standard Class,Australia,Oceania


In [89]:
inventory = (
    df[
        [
            "product_id",
            "warehouse",
            "inventory_level",
            "reorder_level",
            "safety_stock",
            "inventory_status"
        ]
    ]
    .drop_duplicates(subset=["product_id"])
    .reset_index(drop=True)
)

inventory.insert(
    0,
    "inventory_id",
    range(1, len(inventory) + 1)
)

inventory.to_csv(
    "C:/Procurement-Supplier-Performance-Analytics/data/sql/inventory.csv",
    index=False
)

In [90]:
print(inventory.shape)
inventory.head()

(118, 7)


,inventory_id,product_id,warehouse,inventory_level,reorder_level,safety_stock,inventory_status
0,1,1360,Bengaluru,442,89,78,Sufficient
1,2,365,Hyderabad,190,52,22,Sufficient
2,3,627,Hyderabad,435,77,46,Sufficient
3,4,502,Chennai,92,90,64,Sufficient
4,5,278,Mumbai,306,45,42,Sufficient


In [91]:
payments = (
    df[
        [
            "order_id",
            "payment_status",
            "order_date"
        ]
    ]
    .copy()
)

payments["payment_date"] = (
    pd.to_datetime(payments["order_date"])
    + pd.to_timedelta(
        np.random.randint(7, 31, len(payments)),
        unit="D"
    )
)

payments.drop(columns="order_date", inplace=True)

payments.insert(
    0,
    "payment_id",
    range(1, len(payments) + 1)
)

payments.to_csv(
    "C:/Procurement-Supplier-Performance-Analytics/data/sql/payments.csv",
    index=False
)

In [92]:
print(payments.shape)
payments.head()

(180519, 4)


,payment_id,order_id,payment_status,payment_date
0,1,77202,Pending,2018-02-23 22:56:00
1,2,75939,Paid,2018-01-29 12:27:00
2,3,75938,Paid,2018-02-08 12:06:00
3,4,75937,Paid,2018-02-02 11:45:00
4,5,75936,Paid,2018-01-26 11:24:00


In [93]:
print(df["order_id"].duplicated().sum())

114767
